# Preliminary Evaluation — BM25 (Engine A) vs. Neural (Engine B)
**Status update, Oct 25.** Runs both engines on the same 8 queries, pools
their results for group relevance judging, then computes Precision@5,
Recall@5, F1, MAP, and NDCG@5 for each engine side by side.

This is a *preliminary* pass — 8 queries, quick group judging. The final
submission uses the full 20-query suite with two-annotator pooled judging
as described in the proposal.

**How to use this notebook, as a group:**
1. Run Section 1–4 (setup, both engines build their indexes).
2. Run Section 5 — this prints the pooled results for all 8 queries.
3. As a group, fill in the `RELEVANCE` dictionary in Section 6 with 1
   (relevant) or 0 (not relevant) for every `(query, doc_id)` pair shown.
4. Run Section 7 — this computes and prints the final metrics table.


## 1. Setup

In [ ]:
!pip install rank_bm25 sentence-transformers -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


True

In [ ]:
import re
import time
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)-8s]  %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


## 2. Load the shared corpus

Upload `final_news_corpus.csv` into this Colab session (folder icon →
upload) before running this cell.


In [ ]:
def load_mind_corpus(csv_path: str) -> List[Dict[str, str]]:
    path = Path(csv_path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Corpus CSV not found at '{csv_path}'. Upload it to this session first."
        )

    df = pd.read_csv(csv_path, dtype=str)

    required_cols = {"news_id", "title", "abstract", "document_text", "category"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing expected columns: {missing}")

    df["abstract"]      = df["abstract"].fillna("")
    df["document_text"] = df["document_text"].fillna("")
    df = df.rename(columns={"news_id": "id"})

    logger.info("Corpus loaded — %d docs | %d categories", len(df), df["category"].nunique())
    return df.to_dict(orient="records")


CORPUS_CSV = "final_news_corpus.csv"
documents  = load_mind_corpus(CORPUS_CSV)
doc_lookup = {doc["id"]: doc for doc in documents}
print(f"Loaded {len(documents)} documents")


Loaded 10000 documents


## 3. Engine A — BM25 (same code as `engine_a_bm25.ipynb`)

In [ ]:
STOPWORDS = set(stopwords.words("english"))
NEGATION_WORDS = {"not", "no", "nor"}
STOPWORDS -= NEGATION_WORDS

stemmer = PorterStemmer()
_token_pattern = re.compile(r"\b\w+(?:'\w+)?\b")

def preprocess(text: str) -> List[str]:
    text = text.lower()
    tokens = _token_pattern.findall(text)
    return [stemmer.stem(t) for t in tokens if t not in STOPWORDS]


class BM25RetrievalEngine:
    def __init__(self, k1: float = 1.5, b: float = 0.75) -> None:
        self.k1 = k1
        self.b = b
        self.doc_ids: Optional[List[str]] = None
        self.bm25: Optional[BM25Okapi] = None
        self._build_meta: Dict = {}

    def build_index(self, documents, tokenized) -> None:
        t0 = time.perf_counter()
        self.doc_ids = [str(doc["id"]) for doc in documents]
        self.bm25 = BM25Okapi(tokenized, k1=self.k1, b=self.b)
        elapsed = time.perf_counter() - t0
        self._build_meta = {"n_docs": len(documents), "k1": self.k1, "b": self.b,
                             "build_time_s": round(elapsed, 2)}
        logger.info("BM25 index built — %d docs | %.2fs", len(documents), elapsed)

    def search(self, query: str, top_k: int = 10) -> List[Tuple[str, float]]:
        query_tokens = preprocess(query)
        scores = self.bm25.get_scores(query_tokens)
        return self._top_k_results(scores, top_k)

    def _top_k_results(self, scores, top_k: int) -> List[Tuple[str, float]]:
        n = len(self.doc_ids)
        k = min(top_k, n)
        ranked_idx = sorted(range(n), key=lambda i: scores[i], reverse=True)[:k]
        return [(self.doc_ids[i], float(scores[i])) for i in ranked_idx]

    @property
    def index_size_mb(self) -> float:
        # BM25Okapi keeps the inverted index in memory; approximate via corpus token count
        return round(sum(len(t) for t in tokenized_corpus) * 8 / (1024 ** 2), 2)

    def corpus_stats(self) -> Dict:
        return {"model": "BM25Okapi (rank_bm25)", "corpus_size": len(self.doc_ids),
                "index_size_mb_approx": self.index_size_mb, **self._build_meta}


t0 = time.perf_counter()
tokenized_corpus = [preprocess(doc["document_text"]) for doc in documents]
print(f"Tokenised {len(tokenized_corpus)} docs in {time.perf_counter()-t0:.2f}s")

engine_a = BM25RetrievalEngine(k1=1.5, b=0.75)
engine_a.build_index(documents, tokenized_corpus)
print()
for k, v in engine_a.corpus_stats().items():
    print(f"  {k:<22} {v}")


Tokenised 10000 docs in 9.48s

  model                  BM25Okapi (rank_bm25)
  corpus_size            10000
  index_size_mb_approx   2.29
  n_docs                 10000
  k1                     1.5
  b                      0.75
  build_time_s           0.54


## 4. Engine B — Neural (same code as Diveel/Siddu's script)

In [ ]:
class NeuralRetrievalEngine:
    _CACHE_SUFFIX = "_index.npz"

    def __init__(self, model_name="all-MiniLM-L6-v2", cache_dir="./cache",
                 batch_size=128, device=None):
        self.model_name = model_name
        self.cache_dir = Path(cache_dir)
        self.batch_size = batch_size
        _safe = model_name.replace("/", "_").replace("-", "_")
        self.cache_path = self.cache_dir / f"{_safe}{self._CACHE_SUFFIX}"
        self.doc_ids = None
        self.doc_matrix = None
        self._build_meta = {}
        logger.info("Loading model: '%s'", model_name)
        self.model = SentenceTransformer(model_name, device=device)
        if self.cache_path.is_file():
            self._load_index()

    def build_index(self, documents, text_field=None, overwrite=False):
        if self.cache_path.is_file() and not overwrite:
            self._load_index()
            return
        n_docs = len(documents)
        article_ids = [str(doc["id"]) for doc in documents]
        texts = [str(doc[text_field]) for doc in documents] if text_field else \
                [f"{doc['title']} {doc['abstract']}" for doc in documents]
        t0 = time.perf_counter()
        embeddings = self.model.encode(texts, batch_size=self.batch_size,
                                        normalize_embeddings=True,
                                        show_progress_bar=True, convert_to_numpy=True)
        elapsed = time.perf_counter() - t0
        self.doc_ids = np.array(article_ids, dtype=str)
        self.doc_matrix = embeddings.astype(np.float32)
        self._build_meta = {"n_docs": n_docs, "embedding_dim": self.doc_matrix.shape[1],
                             "encode_time_s": round(elapsed, 2)}
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(self.cache_path, doc_ids=self.doc_ids, doc_matrix=self.doc_matrix)
        logger.info("Index cached | shape: %s | %.1fs", self.doc_matrix.shape, elapsed)

    def _load_index(self):
        data = np.load(self.cache_path, allow_pickle=False)
        self.doc_ids = data["doc_ids"]
        self.doc_matrix = data["doc_matrix"]

    def search(self, query: str, top_k: int = 10):
        q_vec = self.model.encode(query, normalize_embeddings=True,
                                   convert_to_numpy=True).astype(np.float32)
        scores = self.doc_matrix @ q_vec
        return self._top_k_results(scores, min(top_k, len(self.doc_ids)))

    def _top_k_results(self, scores, actual_k):
        top_idx = np.argpartition(scores, -actual_k)[-actual_k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        return [(str(self.doc_ids[i]), float(scores[i])) for i in top_idx]

    @property
    def index_size_mb(self):
        return round(self.doc_matrix.nbytes / (1024 ** 2), 2) if self.doc_matrix is not None else 0.0

    def corpus_stats(self):
        return {"model": self.model_name, "corpus_size": len(self.doc_ids),
                "embedding_dim": self.doc_matrix.shape[1], "index_size_mb": self.index_size_mb,
                **self._build_meta}


engine_b = NeuralRetrievalEngine(model_name="all-MiniLM-L6-v2", cache_dir="./cache_b", batch_size=128)
engine_b.build_index(documents, text_field="document_text", overwrite=False)
print()
for k, v in engine_b.corpus_stats().items():
    print(f"  {k:<22} {v}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]


  model                  all-MiniLM-L6-v2
  corpus_size            10000
  embedding_dim          384
  index_size_mb          14.65
  n_docs                 10000
  encode_time_s          427.91


## 5. Run both engines on the 8 preliminary queries, and pool results

For each query: top-5 from BM25 + top-5 from Neural, deduplicated. This
pool is what the group judges in Section 6 — standard pooled-evaluation
practice, exactly as described in the proposal, just at a smaller preliminary
scale (5 per engine instead of 10).


In [ ]:
TOP_K = 5

PRELIM_QUERIES = [
    "professional athlete switches teams",
    "government spending and economic downturn",
    "retirement savings and personal wealth",
    "severe storm causes property destruction",
    "restaurant dining and culinary trends",
    "road trip and outdoor adventure destinations",
    "car model release with advanced technology",
    "patient treatment and hospital procedures",
]

# query -> {"bm25": [(doc_id, score), ...], "neural": [(doc_id, score), ...]}
results = {}
for q in PRELIM_QUERIES:
    results[q] = {
        "bm25":   engine_a.search(q, top_k=TOP_K),
        "neural": engine_b.search(q, top_k=TOP_K),
    }

# Build the pool: every unique doc_id shown by either engine, per query
pools = {}
for q, r in results.items():
    ids = {doc_id for doc_id, _ in r["bm25"]} | {doc_id for doc_id, _ in r["neural"]}
    pools[q] = sorted(ids)

print(f"Pooled results for {len(PRELIM_QUERIES)} queries.\n")
for q in PRELIM_QUERIES:
    print(f'Query: "{q}"')
    print(f"  Pool size: {len(pools[q])} unique documents (up to {2*TOP_K} possible)\n")
    for doc_id in pools[q]:
        doc = doc_lookup.get(doc_id, {})
        cat = doc.get("category", "?")
        text = doc.get("document_text", "")[:90]
        in_bm25   = any(d == doc_id for d, _ in results[q]["bm25"])
        in_neural = any(d == doc_id for d, _ in results[q]["neural"])
        src = ("BM25" if in_bm25 else "    ") + ("+Neural" if in_neural else "")
        print(f"  [{doc_id}] ({src:<11}) [{cat}]  {text}")
    print()


Pooled results for 8 queries.

Query: "professional athlete switches teams"
  Pool size: 9 unique documents (up to 10 possible)

  [N12200] (    +Neural) [sports]  Teams with most (and fewest) in-state players A look at in-state players on rosters.
  [N16972] (BM25+Neural) [autos]  Atlanta Esports team, the Reign, takes major step in Atlanta sports The Atlanta Braves, Fa
  [N18721] (    +Neural) [sports]  Larry Bird Talks About His One Issue With Player Movement Player movement in the NBA is mo
  [N2612] (BM25       ) [news]  Nashville Kurdish professionals group looks to mentor next generation In its third year, a
  [N34501] (BM25       ) [sports]  New Bears center Cody Whitehair promises to 'get better as we go' The Chicago Bears made s
  [N39928] (BM25       ) [sports]  Standout West Sound athletes sign with college teams Some of the best local high school at
  [N40679] (BM25       ) [sports]  Former Seahawks linebacker Lofa Tatupu launches CBD brand Ex-Seahawk Lofa Tatupu has joine

## 6. Group relevance judging — fill this in together

For every `(query, doc_id)` pair printed above, mark **1** if the document
is actually relevant to the query, **0** if not. Judge as a group — read
each snippet, quick yes/no, move on. Use the `doc_id`s exactly as printed
in Section 5.

The cell below auto-generates the dictionary skeleton with every pair set
to `0` — edit the values directly (don't retype the structure).


In [ ]:
RELEVANCE = {}

# professional athlete switches teams
RELEVANCE[('professional athlete switches teams', 'N12200')] = 0
RELEVANCE[('professional athlete switches teams', 'N16972')] = 0
RELEVANCE[('professional athlete switches teams', 'N18721')] = 1
RELEVANCE[('professional athlete switches teams', 'N2612')] = 0
RELEVANCE[('professional athlete switches teams', 'N34501')] = 0
RELEVANCE[('professional athlete switches teams', 'N39928')] = 1
RELEVANCE[('professional athlete switches teams', 'N40679')] = 0
RELEVANCE[('professional athlete switches teams', 'N53047')] = 1
RELEVANCE[('professional athlete switches teams', 'N8031')] = 0

# government spending and economic downturn
RELEVANCE[('government spending and economic downturn', 'N14883')] = 1
RELEVANCE[('government spending and economic downturn', 'N31970')] = 0
RELEVANCE[('government spending and economic downturn', 'N39951')] = 1
RELEVANCE[('government spending and economic downturn', 'N44938')] = 1
RELEVANCE[('government spending and economic downturn', 'N4653')] = 1
RELEVANCE[('government spending and economic downturn', 'N4917')] = 1
RELEVANCE[('government spending and economic downturn', 'N51815')] = 0
RELEVANCE[('government spending and economic downturn', 'N53444')] = 1
RELEVANCE[('government spending and economic downturn', 'N7939')] = 1
RELEVANCE[('government spending and economic downturn', 'N9399')] = 1

# retirement savings and personal wealth
RELEVANCE[('retirement savings and personal wealth', 'N21878')] = 1
RELEVANCE[('retirement savings and personal wealth', 'N30589')] = 1
RELEVANCE[('retirement savings and personal wealth', 'N3243')] = 1
RELEVANCE[('retirement savings and personal wealth', 'N47033')] = 1
RELEVANCE[('retirement savings and personal wealth', 'N53726')] = 1
RELEVANCE[('retirement savings and personal wealth', 'N60218')] = 1
RELEVANCE[('retirement savings and personal wealth', 'N6078')] = 1
RELEVANCE[('retirement savings and personal wealth', 'N63839')] = 1

# severe storm causes property destruction
RELEVANCE[('severe storm causes property destruction', 'N11809')] = 0
RELEVANCE[('severe storm causes property destruction', 'N15166')] = 1
RELEVANCE[('severe storm causes property destruction', 'N18982')] = 1
RELEVANCE[('severe storm causes property destruction', 'N2714')] = 1
RELEVANCE[('severe storm causes property destruction', 'N30968')] = 1
RELEVANCE[('severe storm causes property destruction', 'N32914')] = 0
RELEVANCE[('severe storm causes property destruction', 'N35863')] = 1
RELEVANCE[('severe storm causes property destruction', 'N43040')] = 1
RELEVANCE[('severe storm causes property destruction', 'N47347')] = 1
RELEVANCE[('severe storm causes property destruction', 'N59642')] = 1

# restaurant dining and culinary trends
RELEVANCE[('restaurant dining and culinary trends', 'N22247')] = 1
RELEVANCE[('restaurant dining and culinary trends', 'N30011')] = 1
RELEVANCE[('restaurant dining and culinary trends', 'N37722')] = 1
RELEVANCE[('restaurant dining and culinary trends', 'N40627')] = 1
RELEVANCE[('restaurant dining and culinary trends', 'N49447')] = 1
RELEVANCE[('restaurant dining and culinary trends', 'N52508')] = 1
RELEVANCE[('restaurant dining and culinary trends', 'N59465')] = 1
RELEVANCE[('restaurant dining and culinary trends', 'N60460')] = 1
RELEVANCE[('restaurant dining and culinary trends', 'N7565')] = 1

# road trip and outdoor adventure destinations
RELEVANCE[('road trip and outdoor adventure destinations', 'N11767')] = 1
RELEVANCE[('road trip and outdoor adventure destinations', 'N1435')] = 1
RELEVANCE[('road trip and outdoor adventure destinations', 'N14536')] = 1
RELEVANCE[('road trip and outdoor adventure destinations', 'N18184')] = 0
RELEVANCE[('road trip and outdoor adventure destinations', 'N25474')] = 0
RELEVANCE[('road trip and outdoor adventure destinations', 'N32075')] = 0
RELEVANCE[('road trip and outdoor adventure destinations', 'N32531')] = 1
RELEVANCE[('road trip and outdoor adventure destinations', 'N4331')] = 1
RELEVANCE[('road trip and outdoor adventure destinations', 'N52050')] = 1
RELEVANCE[('road trip and outdoor adventure destinations', 'N52950')] = 0

# car model release with advanced technology
RELEVANCE[('car model release with advanced technology', 'N19204')] = 1
RELEVANCE[('car model release with advanced technology', 'N20643')] = 1
RELEVANCE[('car model release with advanced technology', 'N20854')] = 1
RELEVANCE[('car model release with advanced technology', 'N27025')] = 0
RELEVANCE[('car model release with advanced technology', 'N31915')] = 1
RELEVANCE[('car model release with advanced technology', 'N38092')] = 1
RELEVANCE[('car model release with advanced technology', 'N41407')] = 0
RELEVANCE[('car model release with advanced technology', 'N58706')] = 0
RELEVANCE[('car model release with advanced technology', 'N64908')] = 0
RELEVANCE[('car model release with advanced technology', 'N6492')] = 0

# patient treatment and hospital procedures
RELEVANCE[('patient treatment and hospital procedures', 'N14516')] = 1
RELEVANCE[('patient treatment and hospital procedures', 'N19')] = 1
RELEVANCE[('patient treatment and hospital procedures', 'N38907')] = 1
RELEVANCE[('patient treatment and hospital procedures', 'N42672')] = 1
RELEVANCE[('patient treatment and hospital procedures', 'N50097')] = 1
RELEVANCE[('patient treatment and hospital procedures', 'N57709')] = 1
RELEVANCE[('patient treatment and hospital procedures', 'N59517')] = 0
RELEVANCE[('patient treatment and hospital procedures', 'N64851')] = 1

## 7. Compute metrics

Precision@5, Recall@5 (relative to the pool, standard for pooled
evaluation), F1, Average Precision, and NDCG@5 — for both engines,
per query and averaged.


In [ ]:
import math

def dcg_at_k(relevances: List[int], k: int) -> float:
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances[:k]))

def ndcg_at_k(ranked_doc_ids: List[str], relevance_lookup: Dict[str, int], k: int) -> float:
    gains = [relevance_lookup.get(d, 0) for d in ranked_doc_ids]
    dcg = dcg_at_k(gains, k)
    ideal = dcg_at_k(sorted(gains, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0

def average_precision(ranked_doc_ids: List[str], relevance_lookup: Dict[str, int]) -> float:
    hits, score = 0, 0.0
    for i, d in enumerate(ranked_doc_ids, 1):
        if relevance_lookup.get(d, 0) > 0:
            hits += 1
            score += hits / i
    total_relevant = sum(1 for v in relevance_lookup.values() if v > 0)
    return score / total_relevant if total_relevant > 0 else 0.0


def evaluate_engine(engine_key: str) -> Dict[str, float]:
    precisions, recalls, f1s, aps, ndcgs = [], [], [], [], []
    for q in PRELIM_QUERIES:
        ranked = [doc_id for doc_id, _ in results[q][engine_key]]
        rel_lookup = {doc_id: RELEVANCE.get((q, doc_id), 0) for doc_id in pools[q]}
        total_relevant_in_pool = sum(1 for v in rel_lookup.values() if v > 0)

        hits = sum(1 for d in ranked if rel_lookup.get(d, 0) > 0)
        precision = hits / len(ranked) if ranked else 0.0
        recall = hits / total_relevant_in_pool if total_relevant_in_pool > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        aps.append(average_precision(ranked, rel_lookup))
        ndcgs.append(ndcg_at_k(ranked, rel_lookup, TOP_K))

    return {
        "Precision@5": sum(precisions) / len(precisions),
        "Recall@5":    sum(recalls) / len(recalls),
        "F1@5":        sum(f1s) / len(f1s),
        "MAP":         sum(aps) / len(aps),
        "NDCG@5":      sum(ndcgs) / len(ndcgs),
    }


bm25_metrics   = evaluate_engine("bm25")
neural_metrics = evaluate_engine("neural")

print(f"{'Metric':<14}{'BM25':>10}{'Neural':>10}")
print("-" * 34)
for metric in bm25_metrics:
    print(f"{metric:<14}{bm25_metrics[metric]:>10.3f}{neural_metrics[metric]:>10.3f}")

print()
print(f"{'':14}{'BM25':>10}{'Neural':>10}")
print(f"{'Build time (s)':<14}{engine_a.corpus_stats().get('build_time_s','-'):>10}"
      f"{engine_b.corpus_stats().get('encode_time_s','-'):>10}")
print(f"{'Index size (MB)':<14}{engine_a.corpus_stats().get('index_size_mb_approx','-'):>10}"
      f"{engine_b.corpus_stats().get('index_size_mb','-'):>10}")


Metric              BM25    Neural
----------------------------------
Precision@5        0.625     0.850
Recall@5           0.427     0.654
F1@5               0.499     0.722
MAP                0.410     0.608
NDCG@5             0.837     0.953

                    BM25    Neural
Build time (s)      0.54    457.34
Index size (MB)      2.29     14.65
